# Detección Automática de Zonas Deforestadas en Colombia
### Imágenes Satelitales y Deep Learning como Herramienta para Política Pública

**Curso:** Deep Learning — Maestría en Inteligencia Analítica de Datos (MIAD)  
**Repositorio:** [Andres-Acuna/dl_deforestation](https://github.com/Andres-Acuna/dl_deforestation)

---

Este notebook documenta el desarrollo completo del proyecto, desde la adquisición de datos hasta la generación de mapas de predicción. El flujo está organizado de forma secuencial, con controles para evitar reprocesamiento de pasos ya ejecutados.

**Fuentes de datos:**
- Planet Amazon (Kaggle) — entrenamiento del backbone
- Sentinel-2 via Google Earth Engine — imágenes colombianas
- IDEAM — rasters de pérdida de bosque como etiquetas

**Arquitectura:** EfficientNet-B4 preentrenado en Planet Amazon, transferido a U-Net para segmentación en Colombia.

---
## 0. Configuración del Entorno

Montaje de Drive, clonación del repositorio e instalación de dependencias. Esta sección debe ejecutarse al inicio de cada sesión.

In [ ]:
import subprocess, sys, os
from pathlib import Path

# Verificar GPU
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True)
if gpu.returncode == 0:
    print(f'GPU: {gpu.stdout.strip()}')
else:
    print('Sin GPU. Cambiar entorno de ejecucion a T4 en: Entorno de ejecucion > Cambiar tipo.')

import torch
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Dispositivo: {torch.cuda.get_device_name(0)}')


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DRIVE_BASE = Path('/content/drive/MyDrive/deforestation_project')
assert DRIVE_BASE.exists(), (
    f'No se encontro la carpeta del proyecto en Drive: {DRIVE_BASE}\n'
    'Crea la estructura de carpetas antes de continuar.'
)
print(f'Drive montado correctamente: {DRIVE_BASE}')


In [ ]:
# Clonar o actualizar repositorio
REPO_URL  = 'https://github.com/Andres-Acuna/dl_deforestation.git'
REPO_PATH = Path('/content/dl_deforestation')

if REPO_PATH.exists():
    result = subprocess.run(['git', 'pull'], cwd=REPO_PATH,
                            capture_output=True, text=True)
    print(f'Repositorio actualizado: {result.stdout.strip()}')
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_PATH)],
                   capture_output=True)
    print(f'Repositorio clonado en {REPO_PATH}')

if str(REPO_PATH) not in sys.path:
    sys.path.insert(0, str(REPO_PATH))


In [ ]:
# Instalacion de dependencias no incluidas en Colab
libs = ['rasterio', 'geopandas', 'albumentations',
        'segmentation-models-pytorch', 'earthengine-api']

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q'] + libs,
    capture_output=True, text=True
)
print('Dependencias instaladas' if result.returncode == 0
      else f'Error: {result.stderr[:300]}')


In [ ]:
# Variables globales del proyecto
# Copiar este bloque al inicio de cada sesion nueva

DATA_DIR      = DRIVE_BASE / 'data'
RAW_PLANET    = DATA_DIR / 'raw' / 'planets-dataset'
RAW_COLOMBIA  = DATA_DIR / 'raw' / 'colombia'
PROC_PLANET   = DATA_DIR / 'processed' / 'planet'
PROC_COLOMBIA = DATA_DIR / 'processed' / 'colombia'
MASKS_DIR     = DATA_DIR / 'masks'
CKPT_DIR      = DRIVE_BASE / 'checkpoints'
LOGS_DIR      = DRIVE_BASE / 'logs'
OUTPUTS_DIR   = DRIVE_BASE / 'outputs'

PLANET_CSV  = RAW_PLANET / 'train_v2.csv'
PLANET_TIF  = RAW_PLANET / 'train-tif-v2'
BEST_CKPT   = CKPT_DIR   / 'planet_best.pt'
COL_CKPT    = CKPT_DIR   / 'colombia_best.pt'

CONFIG = {
    'img_size'       : 224,
    'batch_size'     : 32,
    'num_workers'    : 2,
    'tile_size'      : 224,
    'tile_stride'    : 112,
    'planet_epochs'  : 20,
    'planet_lr'      : 1e-3,
    'num_classes'    : 17,
    'colombia_epochs': 30,
    'colombia_lr'    : 5e-4,
    'seed'           : 42,
}

PLANET_TAGS = [
    'agriculture', 'artisinal_mine', 'bare_ground', 'blooming',
    'blow_down', 'clear', 'cloudy', 'conventional_mine', 'cultivation',
    'habitation', 'haze', 'partly_cloudy', 'primary', 'road',
    'selective_logging', 'slash_burn', 'water'
]
DEFORESTATION_TAGS = {
    'agriculture', 'slash_burn', 'habitation', 'cultivation',
    'bare_ground', 'artisinal_mine', 'selective_logging', 'blow_down'
}

torch.manual_seed(CONFIG['seed'])
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Rutas del proyecto:')
for name, p in [('Planet raw', RAW_PLANET), ('Colombia raw', RAW_COLOMBIA),
                ('Mascaras', MASKS_DIR), ('Checkpoints', CKPT_DIR),
                ('Outputs', OUTPUTS_DIR)]:
    estado = 'OK' if Path(p).exists() else 'NO ENCONTRADA'
    print(f'  [{estado}] {name:<18} {p}')

print(f'\nDispositivo: {DEVICE}')


---
## 1. Dataset Planet Amazon

El dataset Planet Amazon contiene aproximadamente 40.000 imágenes multiespectrales de cuatro bandas (Azul, Verde, Rojo, NIR) capturadas sobre el Amazonas brasileño. Cada imagen mide 256×256 píxeles y está etiquetada con múltiples clases de cobertura terrestre, incluyendo bosque primario, agricultura, tala y quema, entre otras.

Este dataset sirve como fuente de preentrenamiento para el backbone del modelo. El objetivo en esta fase no es detectar deforestación directamente, sino que la red aprenda representaciones espectrales robustas de regiones forestales y perturbadas.

### 1.1 Descompresión del archivo ZIP

In [ ]:
import zipfile

ZIP_PATH = RAW_PLANET / 'planets-dataset.zip'

def contar_imagenes(directorio):
    tifs = list(Path(directorio).rglob('*.tif'))
    jpgs = list(Path(directorio).rglob('*.jpg'))
    return len(tifs) + len(jpgs)

n_existentes = contar_imagenes(RAW_PLANET)

if n_existentes > 0:
    print(f'Dataset ya descomprimido ({n_existentes:,} imagenes encontradas). '
          f'Se omite este paso.')

elif not ZIP_PATH.exists():
    print(f'Archivo no encontrado: {ZIP_PATH}')
    print('Sube el ZIP a Drive en la ruta indicada antes de continuar.')

else:
    print(f'Descomprimiendo {ZIP_PATH.name} '
          f'({ZIP_PATH.stat().st_size / 1e9:.2f} GB)...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        archivos = z.namelist()
        total = len(archivos)
        for i, archivo in enumerate(archivos):
            z.extract(archivo, RAW_PLANET)
            if i % 5000 == 0 and i > 0:
                print(f'  {i:,}/{total:,} ({i/total*100:.1f}%)')
    n_final = contar_imagenes(RAW_PLANET)
    print(f'Descompresion completa: {n_final:,} imagenes.')


### 1.2 Carga y exploración del CSV de etiquetas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# Buscar CSV independientemente de la estructura interna del ZIP
csv_candidatos = list(RAW_PLANET.rglob('train_v2.csv'))
assert len(csv_candidatos) > 0, 'No se encontro train_v2.csv. Verifica la descompresion.'
CSV_PATH = csv_candidatos[0]

df = pd.read_csv(CSV_PATH)
df['tags_list'] = df['tags'].str.split()

print(f'Total de imagenes: {len(df):,}')
print(f'Columnas: {list(df.columns)}')
print()
display(df.head())


In [ ]:
# Distribucion de etiquetas
all_tags = [t for tags in df['tags_list'] for t in tags]
conteo = Counter(all_tags)

fig, ax = plt.subplots(figsize=(10, 5))
tags_ordenados = sorted(conteo.items(), key=lambda x: -x[1])
nombres = [t[0] for t in tags_ordenados]
valores = [t[1] for t in tags_ordenados]

colores = ['#c0392b' if n in DEFORESTATION_TAGS else '#2e7d32' for n in nombres]
ax.barh(nombres, valores, color=colores)
ax.set_xlabel('Frecuencia')
ax.set_title('Distribucion de etiquetas — Planet Amazon')
ax.invert_yaxis()

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#c0392b', label='Relacionada con deforestacion'),
    Patch(color='#2e7d32', label='Otras clases')
], loc='lower right')

plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'distribucion_etiquetas.png', dpi=120)
plt.show()

# Porcentaje de imagenes con al menos una etiqueta de deforestacion
df['es_deforestada'] = df['tags_list'].apply(
    lambda t: int(bool(set(t) & DEFORESTATION_TAGS))
)
print(f'Imagenes con deforestacion: {df["es_deforestada"].sum():,} '
      f'({df["es_deforestada"].mean():.1%})')


### 1.3 Exploración visual de las imágenes

In [ ]:
import rasterio

# Buscar carpeta de TIFs
tif_candidatos = list(RAW_PLANET.rglob('train-tif-v2'))
assert len(tif_candidatos) > 0, 'No se encontro la carpeta train-tif-v2.'
TIF_DIR = tif_candidatos[0]

# Verificar metadatos de una imagen
muestra = sorted(TIF_DIR.glob('*.tif'))[0]
with rasterio.open(muestra) as src:
    print(f'Bandas    : {src.count}  (esperado: 4)')
    print(f'Dimension : {src.height} x {src.width} px')
    print(f'Tipo dato : {src.dtypes[0]}')
    print(f'CRS       : {src.crs}')
    print()
    for i, nombre in enumerate(['Azul', 'Verde', 'Rojo', 'NIR'], start=1):
        banda = src.read(i).astype(np.float32)
        print(f'  Banda {i} {nombre:<6}  '
              f'min={banda.min():.0f}  max={banda.max():.0f}  '
              f'media={banda.mean():.0f}')


In [ ]:
# Visualizacion en color verdadero de una muestra representativa
def normalizar(banda):
    p2, p98 = np.percentile(banda, [2, 98])
    return np.clip((banda - p2) / (p98 - p2 + 1e-6), 0, 1)

def leer_rgb(ruta):
    with rasterio.open(ruta) as src:
        r = normalizar(src.read(3).astype(np.float32))
        g = normalizar(src.read(2).astype(np.float32))
        b = normalizar(src.read(1).astype(np.float32))
    return np.stack([r, g, b], axis=-1)

# Seleccionar muestras de clases distintas
clases_muestra = ['primary', 'agriculture', 'slash_burn',
                  'habitation', 'bare_ground', 'water']
muestras = []
for clase in clases_muestra:
    filas = df[df['tags_list'].apply(lambda t: clase in t)]
    if len(filas) > 0:
        nombre = filas.iloc[0]['image_name']
        ruta = TIF_DIR / f'{nombre}.tif'
        if ruta.exists():
            muestras.append((clase, ruta))

fig, axes = plt.subplots(2, 3, figsize=(13, 8))
fig.suptitle('Muestras del dataset — color verdadero (RGB)', fontsize=13)

for ax, (clase, ruta) in zip(axes.flat, muestras):
    ax.imshow(leer_rgb(ruta))
    color = '#c0392b' if clase in DEFORESTATION_TAGS else '#2e7d32'
    ax.set_title(clase, fontsize=10, color=color, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'muestras_planet.png', dpi=120)
plt.show()


---
## 2. Imágenes Sentinel-2 para Colombia (Google Earth Engine)

Para aplicar el modelo a Colombia se requieren imágenes Sentinel-2 alineadas geográficamente con los rasters del IDEAM. Google Earth Engine permite exportar composiciones medianas anuales directamente a Google Drive, resolviendo los problemas de descarga incompleta que ocurren al intentar obtener escenas individuales desde el portal de Copernicus.

Las áreas de interés corresponden a departamentos con alta presión de deforestación según los reportes del IDEAM: **Caquetá, Meta, Putumayo y Guaviare**.

> **Nota:** Esta sección requiere una cuenta de Google Earth Engine activa. El registro es gratuito para investigación y educacion en: https://earthengine.google.com/signup/

In [ ]:
import ee

# Autenticacion (solo la primera vez por cuenta)
try:
    ee.Initialize()
    print('Earth Engine inicializado correctamente.')
except Exception:
    print('Autenticando por primera vez...')
    ee.Authenticate()
    ee.Initialize()
    print('Autenticacion completada.')


In [ ]:
def enmascarar_nubes_s2(imagen):
    """Elimina pixeles con nubes usando la banda SCL de Sentinel-2 SR."""
    scl = imagen.select('SCL')
    mascara_valida = (scl.neq(3)   # sombra de nube
                       .And(scl.neq(8))   # nube media probabilidad
                       .And(scl.neq(9))   # nube alta probabilidad
                       .And(scl.neq(10))) # cirrus
    return imagen.updateMask(mascara_valida)


def exportar_sentinel2(nombre_aoi, coordenadas, anio, carpeta_drive):
    """
    Exporta una composicion mediana anual de Sentinel-2 a Google Drive.

    Parametros
    ----------
    nombre_aoi   : str   — identificador del area de interes
    coordenadas  : list  — lista de [lon, lat] que define el poligono
    anio         : int   — año de la composicion
    carpeta_drive: str   — carpeta destino en Google Drive
    """
    aoi = ee.Geometry.Polygon([coordenadas])

    coleccion = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterDate(f'{anio}-01-01', f'{anio}-12-31')
        .filterBounds(aoi)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
        .map(enmascarar_nubes_s2)
        .select(['B2', 'B3', 'B4', 'B8'])  # Azul, Verde, Rojo, NIR
        .median()
        .clip(aoi)
    )

    tarea = ee.batch.Export.image.toDrive(
        image=coleccion,
        description=f's2_{nombre_aoi}_{anio}',
        folder=carpeta_drive,
        scale=10,
        region=aoi,
        maxPixels=1e10,
        fileFormat='GeoTIFF',
        formatOptions={'cloudOptimized': True}
    )
    tarea.start()
    return tarea


# Areas de interes — departamentos con alta deforestacion
AREAS_INTERES = {
    'caqueta': [
        [-76.5, 0.5], [-73.0, 0.5], [-73.0, 2.5], [-76.5, 2.5]
    ],
    'meta': [
        [-74.9, 2.9], [-72.0, 2.9], [-72.0, 4.5], [-74.9, 4.5]
    ],
    'putumayo': [
        [-77.0, 0.0], [-74.5, 0.0], [-74.5, 1.5], [-77.0, 1.5]
    ],
}


In [ ]:
# Verificar si las imagenes ya fueron descargadas
tifs_colombia = list(RAW_COLOMBIA.rglob('*.tif'))

if len(tifs_colombia) > 0:
    print(f'Imagenes colombianas ya presentes ({len(tifs_colombia)} archivos). '
          'Se omite la exportacion.')
    for f in tifs_colombia:
        print(f'  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)')

else:
    print('Iniciando exportacion de imagenes Sentinel-2...')
    print('Los archivos apareceran en Drive cuando las tareas de GEE terminen.')
    print('Monitoreo en: https://code.earthengine.google.com/tasks\n')

    tareas = {}
    for nombre, coords in AREAS_INTERES.items():
        t = exportar_sentinel2(
            nombre_aoi=nombre,
            coordenadas=coords,
            anio=2022,
            carpeta_drive='deforestation_project/data/raw/colombia'
        )
        tareas[nombre] = t
        print(f'  Tarea iniciada: s2_{nombre}_2022  (id: {t.id})')

    print('\nEsperar a que las tareas terminen antes de continuar.')


---
## 3. Rasters del IDEAM — Etiquetas de Deforestación

Los rasters del IDEAM representan la pérdida de cobertura boscosa detectada por año. Para usarlos como etiquetas de entrenamiento, deben reproyectarse al mismo sistema de referencia y resolución espacial que las imágenes Sentinel-2 descargadas.

El producto utilizado es el **Mapa de Pérdida de Bosque**, disponible en el portal de Monitoreo de Bosques del IDEAM: http://smbyc.ideam.gov.co/MonitoreoBC-WEB/

> **Archivo esperado:** un GeoTIFF por área de interés, en `data/masks/`, con valores 0 (no deforestado) y 1 (deforestado) o valores del año de detección.

In [ ]:
import rasterio
from rasterio.warp import reproject, Resampling

def alinear_mascara_con_imagen(ruta_mascara, ruta_imagen_ref, ruta_salida):
    """
    Reproyecta y reamuestra el raster del IDEAM al CRS y resolucion
    de la imagen Sentinel-2 de referencia.

    Usa Resampling.nearest para preservar los valores categoricos
    de la mascara sin interpolacion.
    """
    with rasterio.open(ruta_imagen_ref) as ref:
        crs_destino   = ref.crs
        transform_dst = ref.transform
        ancho         = ref.width
        alto          = ref.height

    with rasterio.open(ruta_mascara) as src:
        datos = np.zeros((1, alto, ancho), dtype=np.uint8)
        reproject(
            source=rasterio.band(src, 1),
            destination=datos,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform_dst,
            dst_crs=crs_destino,
            resampling=Resampling.nearest
        )

    # Binarizar: cualquier año de deteccion >= 1 es deforestacion
    datos_binarios = (datos > 0).astype(np.uint8)

    with rasterio.open(
        ruta_salida, 'w', driver='GTiff',
        height=alto, width=ancho, count=1,
        dtype=np.uint8, crs=crs_destino, transform=transform_dst
    ) as dst:
        dst.write(datos_binarios)

    print(f'Mascara alineada guardada en: {ruta_salida}')
    area_def = datos_binarios.sum() * 100 / 10000  # pixeles * 100m2 / 10000 = ha
    print(f'Area deforestada detectada: {area_def:,.0f} ha (aprox.)')


# Procesar mascaras disponibles
mascaras_raw = list(MASKS_DIR.rglob('*ideam*.tif')) + \
               list(MASKS_DIR.rglob('*bosque*.tif')) + \
               list(MASKS_DIR.rglob('*deforest*.tif'))

if len(mascaras_raw) == 0:
    print('No se encontraron rasters del IDEAM en:', MASKS_DIR)
    print('Descarga el producto desde: http://smbyc.ideam.gov.co/MonitoreoBC-WEB/')
    print('y coloca los archivos .tif en la carpeta data/masks/')
else:
    imagenes_colombia = list(RAW_COLOMBIA.rglob('*.tif'))
    if len(imagenes_colombia) == 0:
        print('Primero completa la Seccion 2 (exportacion GEE).')
    else:
        for mascara in mascaras_raw:
            nombre = mascara.stem
            # Buscar imagen de referencia con nombre similar
            refs = [i for i in imagenes_colombia if nombre.split('_')[1] in i.stem]
            if refs:
                salida = MASKS_DIR / f'{nombre}_alineada.tif'
                if not salida.exists():
                    alinear_mascara_con_imagen(mascara, refs[0], salida)
                else:
                    print(f'Mascara ya procesada: {salida.name}')


---
## 4. Pipeline de Datos

Se definen los datasets y dataloaders para las dos fases del entrenamiento. El dataset de Planet Amazon se usa para clasificación multi-label. El dataset colombiano, construido a partir de tiles generados sobre las imágenes Sentinel-2 con sus máscaras correspondientes, se usa para segmentación binaria.

### 4.1 Dataset Planet Amazon

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

TAG_TO_IDX = {tag: i for i, tag in enumerate(PLANET_TAGS)}


def get_transforms_planet(modo='train'):
    comunes = [
        A.Normalize(
            mean=[0.485, 0.456, 0.406, 0.5],
            std=[0.229, 0.224, 0.225, 0.25]
        ),
        ToTensorV2()
    ]
    if modo == 'train':
        return A.Compose([
            A.RandomCrop(CONFIG['img_size'], CONFIG['img_size']),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.OneOf([A.GaussNoise(p=1), A.GaussianBlur(p=1)], p=0.2),
            A.ColorJitter(brightness=0.2, contrast=0.2, p=0.3),
        ] + comunes)
    return A.Compose([A.CenterCrop(CONFIG['img_size'], CONFIG['img_size'])] + comunes)


class PlanetDataset(Dataset):
    def __init__(self, df, tif_dir, modo='train'):
        n = len(df)
        if modo == 'train':
            self.df = df.iloc[:int(n * 0.8)].reset_index(drop=True)
        elif modo == 'val':
            self.df = df.iloc[int(n * 0.8):int(n * 0.9)].reset_index(drop=True)
        else:
            self.df = df.iloc[int(n * 0.9):].reset_index(drop=True)

        self.tif_dir  = Path(tif_dir)
        self.transform = get_transforms_planet(modo)

    def _codificar_etiquetas(self, tags):
        vec = np.zeros(len(PLANET_TAGS), dtype=np.float32)
        for t in tags:
            if t in TAG_TO_IDX:
                vec[TAG_TO_IDX[t]] = 1.0
        return vec

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        fila   = self.df.iloc[idx]
        ruta   = self.tif_dir / f'{fila["image_name"]}.tif'
        etiq   = self._codificar_etiquetas(fila['tags_list'])

        with rasterio.open(ruta) as src:
            img = src.read([1, 2, 3, 4]).astype(np.float32)
        img = np.transpose(img, (1, 2, 0))
        img = np.clip(img / 10000.0, 0, 1)

        aug = self.transform(image=img)
        return aug['image'], torch.tensor(etiq)


In [ ]:
# Construir dataloaders Planet
ds_train = PlanetDataset(df, TIF_DIR, modo='train')
ds_val   = PlanetDataset(df, TIF_DIR, modo='val')
ds_test  = PlanetDataset(df, TIF_DIR, modo='test')

dl_train = DataLoader(ds_train, batch_size=CONFIG['batch_size'],
                      shuffle=True,  num_workers=CONFIG['num_workers'], pin_memory=True)
dl_val   = DataLoader(ds_val,   batch_size=CONFIG['batch_size'],
                      shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=True)
dl_test  = DataLoader(ds_test,  batch_size=CONFIG['batch_size'],
                      shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=True)

print(f'Train : {len(ds_train):,} muestras  ({len(dl_train)} batches)')
print(f'Val   : {len(ds_val):,} muestras  ({len(dl_val)} batches)')
print(f'Test  : {len(ds_test):,} muestras  ({len(dl_test)} batches)')

# Verificar un batch
imgs, labels = next(iter(dl_train))
print(f'\nBatch de verificacion:')
print(f'  Imagenes : {imgs.shape}  dtype={imgs.dtype}')
print(f'  Etiquetas: {labels.shape}  dtype={labels.dtype}')


### 4.2 Generación de tiles — Colombia

In [ ]:
def generar_tiles(ruta_imagen, ruta_mascara, dir_salida,
                  tam_tile=224, stride=112, min_pixeles_validos=0.8):
    """
    Divide una imagen GeoTIFF y su mascara en parches de tam_tile x tam_tile px.
    El stride menor al tamano genera solapamiento entre tiles adyacentes,
    lo que ayuda a la cobertura de bordes durante el entrenamiento.

    Los tiles se guardan como arrays NumPy en subdirectorios 'images' y 'masks'.
    """
    dir_salida = Path(dir_salida)
    (dir_salida / 'images').mkdir(parents=True, exist_ok=True)
    (dir_salida / 'masks').mkdir(parents=True, exist_ok=True)

    with rasterio.open(ruta_imagen) as img_src, \
         rasterio.open(ruta_mascara) as mask_src:
        H, W = img_src.height, img_src.width
        n_generados, n_descartados = 0, 0

        for fila in range(0, H - tam_tile, stride):
            for col in range(0, W - tam_tile, stride):
                ventana = rasterio.windows.Window(col, fila, tam_tile, tam_tile)
                tile_img  = img_src.read(window=ventana).astype(np.float32)
                tile_mask = mask_src.read(1, window=ventana).astype(np.uint8)

                # Descartar tiles con exceso de nodata
                if np.isfinite(tile_img).all(axis=0).mean() < min_pixeles_validos:
                    n_descartados += 1
                    continue

                np.save(dir_salida / 'images' / f'tile_{n_generados:05d}.npy', tile_img)
                np.save(dir_salida / 'masks'  / f'tile_{n_generados:05d}.npy', tile_mask)
                n_generados += 1

    print(f'  Tiles generados  : {n_generados:,}')
    print(f'  Tiles descartados: {n_descartados:,}')
    return n_generados


# Generar tiles para cada par imagen/mascara disponible
imagenes_col  = sorted(RAW_COLOMBIA.rglob('s2_*.tif'))
mascaras_col  = sorted(MASKS_DIR.rglob('*_alineada.tif'))

if len(imagenes_col) == 0 or len(mascaras_col) == 0:
    print('Seccion pendiente: requiere completar Secciones 2 y 3 primero.')
else:
    tiles_existentes = list((PROC_COLOMBIA / 'images').rglob('*.npy'))
    if len(tiles_existentes) > 0:
        print(f'Tiles ya generados ({len(tiles_existentes):,}). Se omite este paso.')
    else:
        print('Generando tiles...')
        for img_path, mask_path in zip(imagenes_col, mascaras_col):
            print(f'  Procesando: {img_path.name}')
            generar_tiles(img_path, mask_path, PROC_COLOMBIA,
                          tam_tile=CONFIG['tile_size'],
                          stride=CONFIG['tile_stride'])


### 4.3 Dataset Colombia

In [ ]:
class ColombiaDataset(Dataset):
    def __init__(self, tiles_dir, modo='train'):
        self.rutas = sorted((Path(tiles_dir) / 'images').glob('*.npy'))
        n = len(self.rutas)
        if modo == 'train':
            self.rutas = self.rutas[:int(n * 0.70)]
        elif modo == 'val':
            self.rutas = self.rutas[int(n * 0.70):int(n * 0.85)]
        else:
            self.rutas = self.rutas[int(n * 0.85):]

        comunes = [
            A.Normalize(mean=[0.485, 0.456, 0.406, 0.5],
                        std=[0.229, 0.224, 0.225, 0.25]),
            ToTensorV2()
        ]
        if modo == 'train':
            self.transform = A.Compose([
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.5),
                A.RandomRotate90(p=0.5),
                A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, p=0.3),
            ] + comunes)
        else:
            self.transform = A.Compose(comunes)

    def __len__(self):
        return len(self.rutas)

    def __getitem__(self, idx):
        ruta_img  = self.rutas[idx]
        ruta_mask = ruta_img.parent.parent / 'masks' / ruta_img.name

        img  = np.load(ruta_img)
        mask = np.load(ruta_mask)

        img  = np.clip(img / 10000.0, 0, 1)
        img  = np.transpose(img, (1, 2, 0))

        aug = self.transform(image=img, mask=mask.astype(np.float32))
        return aug['image'], aug['mask'].long()


---
## 5. Arquitectura del Modelo

La arquitectura sigue una estrategia de transferencia en dos fases. En la primera fase se entrena un backbone EfficientNet-B4 con un head de clasificación multi-label sobre el dataset Planet Amazon. En la segunda fase, los pesos del backbone se transfieren a una arquitectura U-Net para segmentación pixel a pixel sobre las imágenes colombianas.

La primera capa convolucional se adapta de 3 a 4 canales de entrada. Los pesos de los tres canales RGB se cargan desde ImageNet; el canal NIR se inicializa como el promedio de los otros tres, una práctica estándar en teledetección.

In [ ]:
import torch.nn as nn
import torchvision.models as tv_models

class BackboneDeforestacion(nn.Module):
    """EfficientNet-B4 adaptado a 4 canales de entrada."""

    def __init__(self, pretrained=True, canales_entrada=4):
        super().__init__()
        weights = tv_models.EfficientNet_B4_Weights.DEFAULT if pretrained else None
        base    = tv_models.efficientnet_b4(weights=weights)

        conv_original = base.features[0][0]
        conv_nueva = nn.Conv2d(
            canales_entrada,
            conv_original.out_channels,
            kernel_size=conv_original.kernel_size,
            stride=conv_original.stride,
            padding=conv_original.padding,
            bias=conv_original.bias is not None
        )

        if pretrained:
            with torch.no_grad():
                conv_nueva.weight[:, :3] = conv_original.weight
                conv_nueva.weight[:, 3]  = conv_original.weight.mean(dim=1)

        base.features[0][0] = conv_nueva
        self.features     = base.features
        self.avgpool      = base.avgpool
        self.feature_dim  = base.classifier[1].in_features  # 1792

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        return torch.flatten(x, 1)


class ModeloPlanet(nn.Module):
    """Backbone + head de clasificacion multi-label para Planet Amazon."""

    def __init__(self, pretrained=True, dropout=0.3):
        super().__init__()
        self.backbone = BackboneDeforestacion(pretrained=pretrained)
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(self.backbone.feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(512, CONFIG['num_classes'])
        )

    def forward(self, x):
        return self.head(self.backbone(x))


# Instanciar y verificar forma de salida
modelo_planet = ModeloPlanet(pretrained=True).to(DEVICE)
with torch.no_grad():
    x_prueba = torch.randn(2, 4, CONFIG['img_size'], CONFIG['img_size']).to(DEVICE)
    salida   = modelo_planet(x_prueba)

print(f'Entrada : {x_prueba.shape}')
print(f'Salida  : {salida.shape}  (esperado: [2, {CONFIG["num_classes"]}])')
n_params = sum(p.numel() for p in modelo_planet.parameters() if p.requires_grad)
print(f'Parametros entrenables: {n_params:,}')


---
## 6. Entrenamiento — Fase 1: Planet Amazon

El objetivo de esta fase es que el backbone aprenda representaciones espectrales de cobertura terrestre. Se usa `BCEWithLogitsLoss` con pesos positivos para compensar el desbalance de clases, y `OneCycleLR` como scheduler, que ha demostrado convergencia más rápida en datasets de imágenes satelitales.

La métrica principal es el **F2-score por muestra**, que penaliza más los falsos negativos que los falsos positivos — relevante porque omitir una zona deforestada tiene mayor costo que una falsa alarma.

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
from sklearn.metrics import fbeta_score


def entrenar_planet(modelo, dl_train, dl_val, config, ruta_ckpt):
    pos_weight = torch.ones(config['num_classes']) * 2.0
    criterio   = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(DEVICE))
    optimizador = AdamW(modelo.parameters(),
                        lr=config['planet_lr'], weight_decay=1e-4)
    scheduler  = OneCycleLR(
        optimizador,
        max_lr=config['planet_lr'],
        epochs=config['planet_epochs'],
        steps_per_epoch=len(dl_train)
    )

    mejor_f2 = 0.0
    historial = {'train_loss': [], 'val_f2': []}

    for epoca in range(config['planet_epochs']):
        # Entrenamiento
        modelo.train()
        loss_total = 0.0
        for imgs, labels in dl_train:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizador.zero_grad()
            logits = modelo(imgs)
            loss   = criterio(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
            optimizador.step()
            scheduler.step()
            loss_total += loss.item()

        # Validacion
        modelo.eval()
        preds_all, labels_all = [], []
        with torch.no_grad():
            for imgs, labels in dl_val:
                logits = modelo(imgs.to(DEVICE))
                preds_all.append(torch.sigmoid(logits).cpu().numpy())
                labels_all.append(labels.numpy())

        preds  = (np.vstack(preds_all) > 0.2).astype(int)
        labels = np.vstack(labels_all)
        f2     = fbeta_score(labels, preds, beta=2,
                             average='samples', zero_division=0)

        loss_media = loss_total / len(dl_train)
        historial['train_loss'].append(loss_media)
        historial['val_f2'].append(f2)

        print(f'Epoca {epoca+1:>3}/{config["planet_epochs"]}  '
              f'loss={loss_media:.4f}  F2={f2:.4f}')

        if f2 > mejor_f2:
            mejor_f2 = f2
            torch.save({'epoch': epoca,
                        'model_state_dict': modelo.state_dict(),
                        'f2': mejor_f2}, ruta_ckpt)
            print(f'  Checkpoint guardado (F2={mejor_f2:.4f})')

    return historial


In [ ]:
# Ejecutar entrenamiento Fase 1
# Si ya existe un checkpoint valido, se carga directamente

if BEST_CKPT.exists():
    ckpt = torch.load(BEST_CKPT, map_location=DEVICE)
    modelo_planet.load_state_dict(ckpt['model_state_dict'])
    print(f'Checkpoint cargado: {BEST_CKPT.name}  (F2={ckpt["f2"]:.4f}, epoca={ckpt["epoch"]+1})')
    print('Se omite el entrenamiento. Para re-entrenar, elimina el archivo de checkpoint.')
    historial_planet = None

else:
    print(f'Iniciando entrenamiento en {DEVICE}...')
    CKPT_DIR.mkdir(parents=True, exist_ok=True)
    historial_planet = entrenar_planet(
        modelo_planet, dl_train, dl_val, CONFIG, BEST_CKPT
    )


In [ ]:
# Curvas de aprendizaje (solo si se entrenó en esta sesion)
if historial_planet is not None:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(historial_planet['train_loss'], color='#2e7d32')
    ax1.set_title('Perdida de entrenamiento')
    ax1.set_xlabel('Epoca')
    ax1.set_ylabel('BCE Loss')
    ax1.grid(alpha=0.3)

    ax2.plot(historial_planet['val_f2'], color='#1565c0')
    ax2.set_title('F2-score en validacion')
    ax2.set_xlabel('Epoca')
    ax2.set_ylabel('F2')
    ax2.grid(alpha=0.3)

    plt.suptitle('Entrenamiento Fase 1 — Planet Amazon', fontsize=12)
    plt.tight_layout()
    plt.savefig(OUTPUTS_DIR / 'curvas_fase1.png', dpi=120)
    plt.show()


---
## 7. Fine-tuning — Fase 2: Segmentación en Colombia

En esta fase el backbone preentrenado se integra como encoder de una U-Net. El decoder se inicializa desde cero y se entrena junto con el encoder, aplicando learning rates diferenciales: el encoder recibe un rate diez veces menor que el decoder, para preservar las representaciones aprendidas en Fase 1 sin sobreescribirlas con gradientes ruidosos al inicio del fine-tuning.

La función de pérdida combina Dice Loss y BCE, lo que la hace robusta ante el desbalance espacial típico de estas imágenes (la mayoría de los píxeles son bosque).

In [ ]:
import segmentation_models_pytorch as smp


def construir_modelo_colombia(ruta_ckpt_planet=None):
    """
    Construye una U-Net con encoder EfficientNet-B4 de 4 canales.
    Si se proporciona un checkpoint de Fase 1, intenta cargar los pesos
    del encoder que coincidan por nombre.
    """
    modelo = smp.Unet(
        encoder_name='efficientnet-b4',
        encoder_weights='imagenet',
        in_channels=4,
        classes=1,
        activation=None
    )

    if ruta_ckpt_planet and Path(ruta_ckpt_planet).exists():
        ckpt = torch.load(ruta_ckpt_planet, map_location='cpu')
        estado_planet = ckpt['model_state_dict']

        # Cargar solo los pesos del encoder que coincidan por nombre
        estado_unet = modelo.state_dict()
        cargados = 0
        for k, v in estado_planet.items():
            k_unet = k.replace('backbone.', 'encoder.')
            if k_unet in estado_unet and estado_unet[k_unet].shape == v.shape:
                estado_unet[k_unet] = v
                cargados += 1
        modelo.load_state_dict(estado_unet)
        print(f'Pesos transferidos desde Fase 1: {cargados} capas')
    else:
        print('Usando pesos de ImageNet para el encoder.')

    return modelo


def entrenar_colombia(modelo, dl_train, dl_val, config, ruta_ckpt):
    criterio = (smp.losses.DiceLoss(mode='binary') +
                smp.losses.SoftBCEWithLogitsLoss())

    # Learning rates diferenciales
    optimizador = AdamW([
        {'params': modelo.encoder.parameters(),
         'lr': config['colombia_lr'] * 0.1},
        {'params': modelo.decoder.parameters(),
         'lr': config['colombia_lr']},
        {'params': modelo.segmentation_head.parameters(),
         'lr': config['colombia_lr']},
    ], weight_decay=1e-4)

    mejor_iou = 0.0
    historial = {'train_loss': [], 'val_iou': []}

    for epoca in range(config['colombia_epochs']):
        modelo.train()
        loss_total = 0.0
        for imgs, masks in dl_train:
            imgs  = imgs.to(DEVICE)
            masks = masks.float().unsqueeze(1).to(DEVICE)
            optimizador.zero_grad()
            loss = criterio(modelo(imgs), masks)
            loss.backward()
            optimizador.step()
            loss_total += loss.item()

        modelo.eval()
        ious = []
        with torch.no_grad():
            for imgs, masks in dl_val:
                preds = torch.sigmoid(modelo(imgs.to(DEVICE))) > 0.5
                masks = masks.unsqueeze(1).bool()
                inter = (preds.cpu() & masks).float().sum()
                union = (preds.cpu() | masks).float().sum()
                ious.append((inter / (union + 1e-8)).item())

        iou_media  = float(np.mean(ious))
        loss_media = loss_total / len(dl_train)
        historial['train_loss'].append(loss_media)
        historial['val_iou'].append(iou_media)

        print(f'Epoca {epoca+1:>3}/{config["colombia_epochs"]}  '
              f'loss={loss_media:.4f}  IoU={iou_media:.4f}')

        if iou_media > mejor_iou:
            mejor_iou = iou_media
            torch.save({'epoch': epoca,
                        'model_state_dict': modelo.state_dict(),
                        'iou': mejor_iou}, ruta_ckpt)

    return historial


In [ ]:
tiles_disponibles = list((PROC_COLOMBIA / 'images').rglob('*.npy'))

if len(tiles_disponibles) == 0:
    print('Seccion pendiente: primero completa Secciones 2, 3 y 4 (tiles de Colombia).')

elif COL_CKPT.exists():
    modelo_colombia = construir_modelo_colombia(BEST_CKPT).to(DEVICE)
    ckpt = torch.load(COL_CKPT, map_location=DEVICE)
    modelo_colombia.load_state_dict(ckpt['model_state_dict'])
    print(f'Checkpoint Colombia cargado (IoU={ckpt["iou"]:.4f}, epoca={ckpt["epoch"]+1})')
    historial_colombia = None

else:
    ds_col_train = ColombiaDataset(PROC_COLOMBIA, modo='train')
    ds_col_val   = ColombiaDataset(PROC_COLOMBIA, modo='val')

    dl_col_train = DataLoader(ds_col_train, batch_size=16,
                              shuffle=True, num_workers=CONFIG['num_workers'])
    dl_col_val   = DataLoader(ds_col_val,   batch_size=16,
                              shuffle=False, num_workers=CONFIG['num_workers'])

    modelo_colombia = construir_modelo_colombia(BEST_CKPT).to(DEVICE)
    print(f'Iniciando fine-tuning en {DEVICE}...')
    historial_colombia = entrenar_colombia(
        modelo_colombia, dl_col_train, dl_col_val, CONFIG, COL_CKPT
    )


---
## 8. Evaluación y Resultados

Se evalúa el modelo final sobre el conjunto de prueba con las métricas definidas en la propuesta: IoU, F1 por clase y F2-score global. Adicionalmente se genera un reporte visual de predicciones sobre imágenes reales de Colombia.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import itertools


def evaluar_segmentacion(modelo, dataloader, umbral=0.5):
    """Calcula metricas de segmentacion sobre un dataloader completo."""
    modelo.eval()
    preds_all, masks_all = [], []

    with torch.no_grad():
        for imgs, masks in dataloader:
            logits = modelo(imgs.to(DEVICE))
            preds  = (torch.sigmoid(logits) > umbral).cpu().numpy()
            preds_all.append(preds.flatten())
            masks_all.append(masks.numpy().flatten())

    y_pred = np.concatenate(preds_all)
    y_true = np.concatenate(masks_all)

    print('Reporte de clasificacion:')
    print(classification_report(y_true, y_pred,
                                 target_names=['Bosque', 'Deforestado']))

    # IoU por clase
    for clase, nombre in enumerate(['Bosque', 'Deforestado']):
        inter = ((y_pred == clase) & (y_true == clase)).sum()
        union = ((y_pred == clase) | (y_true == clase)).sum()
        print(f'IoU {nombre:<15}: {inter / (union + 1e-8):.4f}')

    return y_true, y_pred


if COL_CKPT.exists() and len(tiles_disponibles) > 0:
    ds_col_test = ColombiaDataset(PROC_COLOMBIA, modo='test')
    dl_col_test = DataLoader(ds_col_test, batch_size=16, shuffle=False)
    y_true, y_pred = evaluar_segmentacion(modelo_colombia, dl_col_test)
else:
    print('Seccion pendiente: requiere completar el entrenamiento de Fase 2.')


### 8.1 Visualización de predicciones

In [ ]:
def visualizar_predicciones(modelo, dataset, n=4):
    """Muestra imagen, mascara real y prediccion del modelo lado a lado."""
    modelo.eval()
    indices = np.random.choice(len(dataset), n, replace=False)

    fig, axes = plt.subplots(n, 3, figsize=(12, n * 3.5))
    fig.suptitle('Predicciones del modelo — Colombia', fontsize=13)
    titulos = ['Imagen (RGB)', 'Mascara real', 'Prediccion']

    for fila, idx in enumerate(indices):
        img_t, mask_t = dataset[idx]

        with torch.no_grad():
            logit = modelo(img_t.unsqueeze(0).to(DEVICE))
            pred  = (torch.sigmoid(logit) > 0.5).cpu().squeeze().numpy()

        # Desnormalizar para visualizacion
        img_np = img_t.numpy().transpose(1, 2, 0)
        mean   = np.array([0.485, 0.456, 0.406])
        std    = np.array([0.229, 0.224, 0.225])
        rgb    = np.clip(img_np[:, :, :3] * std + mean, 0, 1)

        datos = [rgb, mask_t.numpy(), pred]
        cmaps = [None, 'RdYlGn_r', 'RdYlGn_r']

        for col, (dato, titulo, cmap) in enumerate(zip(datos, titulos, cmaps)):
            axes[fila, col].imshow(dato, cmap=cmap,
                                   vmin=0, vmax=1 if col > 0 else None)
            if fila == 0:
                axes[fila, col].set_title(titulo, fontsize=10)
            axes[fila, col].axis('off')

    plt.tight_layout()
    plt.savefig(OUTPUTS_DIR / 'predicciones_colombia.png', dpi=120)
    plt.show()


if COL_CKPT.exists() and len(tiles_disponibles) > 0:
    visualizar_predicciones(modelo_colombia, ds_col_test, n=4)
else:
    print('Seccion pendiente: requiere completar el entrenamiento de Fase 2.')


---
## 9. Prototipo de Aplicación

Función de inferencia sobre una imagen GeoTIFF arbitraria. Permite cargar cualquier imagen Sentinel-2 de Colombia y generar el mapa de deforestación predicho con el área estimada en hectáreas.

In [ ]:
def inferencia_imagen_completa(ruta_imagen, modelo, tam_tile=224,
                               stride=112, umbral=0.5):
    """
    Aplica el modelo sobre una imagen GeoTIFF completa mediante
    inferencia por parches con promedio en zonas de solapamiento.

    Retorna el mapa de probabilidad y la estimacion de area deforestada.
    """
    transform = A.Compose([
        A.Normalize(mean=[0.485, 0.456, 0.406, 0.5],
                    std=[0.229, 0.224, 0.225, 0.25]),
        ToTensorV2()
    ])

    modelo.eval()

    with rasterio.open(ruta_imagen) as src:
        imagen = src.read([1, 2, 3, 4]).astype(np.float32)
        perfil = src.profile
        resolucion_m = abs(src.transform.a)  # metros por pixel

    _, H, W = imagen.shape
    mapa_prob   = np.zeros((H, W), dtype=np.float32)
    mapa_conteo = np.zeros((H, W), dtype=np.float32)

    for fila in range(0, H - tam_tile + 1, stride):
        for col in range(0, W - tam_tile + 1, stride):
            tile = imagen[:, fila:fila+tam_tile, col:col+tam_tile]
            tile = np.clip(tile / 10000.0, 0, 1)
            tile = np.transpose(tile, (1, 2, 0))

            aug    = transform(image=tile)
            tensor = aug['image'].unsqueeze(0).to(DEVICE)

            with torch.no_grad():
                prob = torch.sigmoid(modelo(tensor)).cpu().squeeze().numpy()

            mapa_prob[fila:fila+tam_tile, col:col+tam_tile]   += prob
            mapa_conteo[fila:fila+tam_tile, col:col+tam_tile] += 1.0

    mapa_conteo = np.maximum(mapa_conteo, 1)
    mapa_prob  /= mapa_conteo
    mapa_binario = (mapa_prob > umbral).astype(np.uint8)

    # Estimar area deforestada en hectareas
    area_ha = mapa_binario.sum() * (resolucion_m ** 2) / 10_000

    return mapa_prob, mapa_binario, area_ha, perfil


def guardar_mapa_geotiff(mapa_binario, perfil, ruta_salida):
    """Guarda el mapa de prediccion como GeoTIFF para uso en QGIS."""
    perfil.update(count=1, dtype=rasterio.uint8)
    with rasterio.open(ruta_salida, 'w', **perfil) as dst:
        dst.write(mapa_binario[np.newaxis, :, :])
    print(f'Mapa guardado en: {ruta_salida}')


# Ejecutar inferencia sobre imagenes colombianas disponibles
imagenes_col = list(RAW_COLOMBIA.rglob('s2_*.tif'))

if not COL_CKPT.exists():
    print('Seccion pendiente: requiere el checkpoint de Fase 2.')

elif len(imagenes_col) == 0:
    print('Seccion pendiente: no hay imagenes Sentinel-2 colombianas en Drive.')

else:
    img_prueba = imagenes_col[0]
    print(f'Aplicando modelo sobre: {img_prueba.name}')

    mapa_prob, mapa_binario, area_ha, perfil = inferencia_imagen_completa(
        img_prueba, modelo_colombia
    )

    print(f'Area estimada deforestada: {area_ha:,.0f} hectareas')

    ruta_salida_tif = OUTPUTS_DIR / f'prediccion_{img_prueba.stem}.tif'
    guardar_mapa_geotiff(mapa_binario, perfil, ruta_salida_tif)

    # Visualizacion del mapa de prediccion
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    im1 = ax1.imshow(mapa_prob, cmap='RdYlGn_r', vmin=0, vmax=1)
    ax1.set_title('Probabilidad de deforestacion')
    plt.colorbar(im1, ax=ax1, fraction=0.03)

    ax2.imshow(mapa_binario, cmap='RdYlGn_r', vmin=0, vmax=1)
    ax2.set_title(f'Mapa binario  ({area_ha:,.0f} ha deforestadas)')

    for ax in [ax1, ax2]:
        ax.axis('off')

    plt.suptitle(img_prueba.stem, fontsize=11)
    plt.tight_layout()
    plt.savefig(OUTPUTS_DIR / f'mapa_{img_prueba.stem}.png', dpi=120)
    plt.show()


---
## Notas de uso

**Estado del pipeline por sección:**

| Sección | Dependencia | Estado |
|---------|-------------|--------|
| 0. Configuracion | Drive + GitHub | Ejecutar cada sesion |
| 1. Planet Amazon | ZIP en Drive | Listo |
| 2. Sentinel-2 GEE | Cuenta GEE | Pendiente |
| 3. Mascaras IDEAM | Rasters descargados | Pendiente |
| 4. Pipeline datos | Secciones 1-3 | Parcial (Planet listo) |
| 5. Modelo | — | Listo |
| 6. Entrenamiento Fase 1 | Dataset Planet | Listo |
| 7. Fine-tuning Colombia | Secciones 2-4 | Pendiente |
| 8. Evaluacion | Seccion 7 | Pendiente |
| 9. Prototipo | Seccion 7 | Pendiente |

**Para guardar cambios al repositorio desde Colab:**
```python
import subprocess
subprocess.run(['git', 'config', 'user.email', 'correo@ejemplo.com'], cwd='/content/dl_deforestation')
subprocess.run(['git', 'config', 'user.name', 'Nombre'], cwd='/content/dl_deforestation')
subprocess.run(['git', 'add', 'notebooks/proyecto_deforestacion.ipynb'], cwd='/content/dl_deforestation')
subprocess.run(['git', 'commit', '-m', 'update: descripcion del cambio'], cwd='/content/dl_deforestation')
subprocess.run(['git', 'push'], cwd='/content/dl_deforestation')
```